# Entrenar oil palm (YOLOv8) — Colab + Google Drive + GPU

Sigue la guía paso a paso en el chat. Resumen:
1. Sube `oil palm.v1i.yolov8.zip` a una carpeta en Drive (ej. `Palm-stalker`)
2. **Runtime → Change runtime type → T4 GPU**
3. Ejecuta todas las celdas en orden
4. Al final `best.pt` queda en Drive y puedes descargarlo a tu Mac

In [ ]:
# === CONFIGURACIÓN: cambia solo si tu carpeta en Drive tiene otro nombre ===
CARPETA_DRIVE = "Palm-stalker"          # carpeta dentro de "Mi unidad"
ZIP_DATASET = "oil palm.v1i.yolov8.zip"  # nombre exacto del archivo en Drive

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil
import os

# force_remount=True si Drive se desconectó en un intento anterior
drive.mount("/content/drive", force_remount=False)

drive_root = Path("/content/drive/MyDrive") / CARPETA_DRIVE
zip_en_drive = drive_root / ZIP_DATASET
zip_local = Path("/content") / "oil_palm_dataset.zip"
dataset_dir = Path("/content/oil-palm-dataset")

print("Buscando en Drive:", zip_en_drive)
if not zip_en_drive.exists():
    raise FileNotFoundError(
        f"No encontré el ZIP.\n"
        f"Crea '{CARPETA_DRIVE}' en Mi unidad y sube: {ZIP_DATASET}\n"
        f"Ruta esperada: {zip_en_drive}"
    )

# IMPORTANTE: no descomprimir desde Drive directamente (error "Transport endpoint is not connected").
# Primero copiar el ZIP al disco local de Colab (/content).
if zip_local.exists():
    zip_local.unlink()

print("Copiando ZIP a disco local de Colab (~390 MB, 2-5 min)...")
shutil.copy2(zip_en_drive, zip_local)
print("Copia OK:", zip_local, "—", round(zip_local.stat().st_size / 1e6, 1), "MB")

if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
dataset_dir.mkdir(parents=True)

print("Descomprimiendo en /content (varios minutos)...")
with zipfile.ZipFile(zip_local, "r") as z:
    z.extractall(dataset_dir)

print("Listo. Dataset en:", dataset_dir)
print("Imágenes train:", len(list((dataset_dir / "train" / "images").glob("*"))))

In [ ]:
# path debe ser la ruta ABSOLUTA del dataset (path: . falla en Colab → busca /content/valid/images)
data_yaml = dataset_dir / "data.yaml"
data_yaml.write_text(f"""path: {dataset_dir}
train: train/images
val: valid/images
test: test/images

nc: 5
names: ["0", "1", "2", "3", "4"]
""")
print(data_yaml.read_text())

# Comprobar que existen las carpetas
for split in ("train", "valid", "test"):
    imgs = dataset_dir / split / "images"
    print(split, ":", len(list(imgs.glob("*"))), "imágenes en", imgs)

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU. Ve a Runtime → Change runtime type → T4 GPU")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=1024,
    batch=16,
    project="/content/runs/detect",
    name="oil-palm",
    exist_ok=True,
)

In [ ]:
from pathlib import Path
import shutil

best_local = Path("/content/runs/detect/oil-palm/weights/best.pt")
if not best_local.exists():
    raise FileNotFoundError("No existe best.pt. ¿Terminó el entrenamiento sin errores?")

# Copiar a Drive vía archivo temporal local (evita el mismo error de montaje)
tmp_drive = Path("/content/best.pt")
shutil.copy2(best_local, tmp_drive)
destino_drive = drive_root / "best.pt"
shutil.copy2(tmp_drive, destino_drive)

print("Copiado a Drive:", destino_drive)
print("Tamaño (MB):", round(destino_drive.stat().st_size / 1e6, 2))

In [ ]:
# Opcional: descargar best.pt directo al Mac desde Colab
from google.colab import files
files.download(str(best_local))

## En tu Mac

Copia `best.pt` desde Drive a `model-trainning/` y ejecuta:

```bash
cd model-trainning
source .venv/bin/activate
python detectar_palmas.py --pesos best.pt
```